# Day 12 · 数据集 v0 交付（M2 验收）

**配套讲义**: `days/day-12.md` ｜ **本地可跑**
今天交付一份别人能拿去训模型的数据集。

## 1. 数据集体检总表

In [ ]:
import sys; sys.path.insert(0, "..")
import json
from collections import Counter
from pathlib import Path

p = Path("../data/processed/sft_train.jsonl")
if not p.exists():
    print("先跑完 Day 11 的打包。")
else:
    rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    print(f"train: {len(rows)} 条")
    n_img = sum(1 for r in rows if r.get("images"))
    n_multi = sum(1 for r in rows if r.get("images") and len(r["images"]) > 1)
    print(f"含图: {n_img}  多图: {n_multi}  纯文本: {len(rows)-n_img}")
    lens = [len(json.dumps(r, ensure_ascii=False)) for r in rows]
    print(f"样本字节长度 p50/p95: {sorted(lens)[len(lens)//2]}/{sorted(lens)[int(len(lens)*.95)]}")

## 2. 实际分布 vs Day 8 计划

In [ ]:
from src.data.taxonomy import TARGET_DISTRIBUTION
if p.exists():
    c = Counter(r.get("intent", "?") for r in rows)
    total = sum(c.values())
    for intent, cnt in c.most_common():
        plan_n = sum(TARGET_DISTRIBUTION.get(intent, {}).values())
        plan_pct = plan_n / sum(sum(v.values()) for v in TARGET_DISTRIBUTION.values())
        print(f"{intent:<12} 实际 {cnt/total:>6.1%}   计划 {plan_pct:>6.1%}   Δ{cnt/total-plan_pct:+.1%}")

## 3. 生成 DATASET_CARD.md

In [ ]:
from src.data.report import generate_card
card = generate_card(data_dir="../data/processed/")
print(card[:800])
print("……")
# 完整文件: data/processed/DATASET_CARD.md
# 「已知缺陷」至少写 3 条，诚实写。这是面试官最看重的部分。

## 4. 上传 HuggingFace（私有）

In [ ]:
UPLOAD = False   # 确认无敏感信息后改 True
REPO = "your-name/cx-sft-v0"        # ← 改成你的
if UPLOAD:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(REPO, private=True, exist_ok=True, repo_type="dataset")
    api.upload_folder(folder_path="../data/processed/", repo_id=REPO, repo_type="dataset")
    print("上传完成:", f"https://huggingface.co/datasets/{REPO}")
else:
    print("上传前最后一遍 check_leakage + grep 敏感词！")

## 5. M2 验收清单
- [ ] 卡片六字段齐全，已知缺陷 ≥3 条
- [ ] HF 私有仓库上传成功
- [ ] 周复盘写完，W2 六天 `[x]`，M2 `[x]`
- [ ] 一句话说清：这数据集适合训什么、不适合训什么